# Computer Exercise 13.7 — Problem 1

> **교재**: Cheney & Kincaid, *Numerical Mathematics and Computing* (7th ed.)
> **단원**: 13.7 Minimization — *Trust-Region Methods: Cauchy Point & Dogleg*
> **풀이 일자**: Day 52
> **언어**: Python 3 (NumPy / pandas / Matplotlib)


## 1. 문제 (원문)

> **1.** Implement a *trust-region* method for unconstrained minimization. At each iterate solve the
> quadratic model subproblem two ways — by the **Cauchy point** (the minimizer of the model along the
> steepest-descent direction inside the trust region) and by the **dogleg** step — and test both on the
> Rosenbrock function $f(x,y)=(1-x)^2+100\,(y-x^2)^2$ starting from $(-1.2,\,1)$. Use the ratio of
> *actual* to *predicted* reduction $\rho_k$ to accept/reject steps and to update the radius $\Delta_k$.
> Report the iteration history and compare the two step strategies.

### 한국어 풀이용 정리
무제약 최소화의 **신뢰영역(trust-region)** 골격을 직접 구현한다. 각 반복에서 2차 모델
$m_k(p)=f_k+g_k^\top p+\tfrac12 p^\top B_k p$ 를 $\lVert p\rVert\le\Delta_k$ 안에서 (근사적으로) 최소화하되,
그 부분문제 해를 **(a) Cauchy 점**과 **(b) dogleg 스텝** 두 방식으로 구한다. 표준 시험함수인
**Rosenbrock** 함수에 $(-1.2,1)$ 에서 출발해 적용하고, **실제감소/예측감소 비율** $\rho_k$ 로
스텝 수락 여부와 반경 $\Delta_k$ 를 갱신한다. 두 전략의 수렴 거동을 비교한다.


## 2. 수학적 배경

### 2.1 신뢰영역 부분문제
반복점 $x_k$ 에서 목적함수를 2차 모델로 근사한다.
$$
m_k(p)=f_k+g_k^\top p+\tfrac12\,p^\top B_k p,\qquad g_k=\nabla f(x_k),\ \ B_k=\nabla^2 f(x_k),
$$
그리고 **신뢰반경** $\Delta_k$ 안에서만 모델을 믿는다.
$$
\min_{p\in\mathbb R^n}\; m_k(p)\quad\text{s.t.}\quad \lVert p\rVert\le \Delta_k. \tag{TRS}
$$

### 2.2 일치비율과 반경 갱신
스텝 $p_k$ 의 품질은 **실제 감소 / 예측 감소** 비율로 측정한다.
$$
\rho_k=\frac{f(x_k)-f(x_k+p_k)}{m_k(0)-m_k(p_k)}.
$$
$\rho_k$ 가 1에 가까우면 모델이 정확한 것이므로 반경을 키우고, 작거나 음수면 반경을 줄이고 스텝을 거부한다.
$$
\boxed{\;\rho_k<\tfrac14:\ \Delta_{k+1}=\tfrac14\Delta_k,\quad
\rho_k>\tfrac34\ \&\ \lVert p_k\rVert=\Delta_k:\ \Delta_{k+1}=\min(2\Delta_k,\Delta_{\max});\;}
$$
스텝은 $\rho_k>\eta\ (\eta\in[0,\tfrac14))$ 일 때만 수락한다.

### 2.3 Cauchy 점
모델을 **최급강하 방향** $-g_k$ 위에서 신뢰영역까지 최소화한 점이다.
$$
p_k^{\mathrm C}=-\tau_k\frac{\Delta_k}{\lVert g_k\rVert}\,g_k,\qquad
\tau_k=\begin{cases}1,& g_k^\top B_k g_k\le 0,\\[4pt]
\min\!\Big(\dfrac{\lVert g_k\rVert^3}{\Delta_k\,g_k^\top B_k g_k},\,1\Big),&\text{otherwise.}\end{cases}
$$
항상 충분감소를 보장하지만 1차(steepest-descent급) 수렴이라 **느리다**.

### 2.4 Dogleg 스텝 ($B_k\succ 0$ 일 때)
완전 뉴턴 스텝 $p^{\mathrm B}=-B_k^{-1}g_k$ 와 steepest-descent 최소점
$p^{\mathrm U}=-\dfrac{g_k^\top g_k}{g_k^\top B_k g_k}\,g_k$ 를 잇는 꺾인 경로
$$
p(\tau)=\begin{cases}\tau\,p^{\mathrm U},&0\le\tau\le1,\\[2pt]p^{\mathrm U}+(\tau-1)(p^{\mathrm B}-p^{\mathrm U}),&1\le\tau\le2,\end{cases}
$$
를 따라 $\lVert p(\tau)\rVert=\Delta_k$ 가 되는 점(또는 내부면 $p^{\mathrm B}$)을 택한다. 뉴턴 정보를 쓰므로
모델이 좋아질수록 **2차 수렴**에 가까워진다. ($B_k$ 가 양정치가 아니면 본 노트북은 Cauchy 점으로 안전하게 후퇴한다.)


## 3. 풀이 흐름

1. **함수 정의**: Rosenbrock $f$, 기울기 $g$, 헤시안 $B$ 를 해석적으로 구현.
2. **Cauchy 점 / dogleg 스텝** 함수 작성. dogleg은 $B\succ0$ 검사(Cholesky) 후, 아니면 Cauchy로 후퇴.
3. **신뢰영역 루프**: $\rho_k$ 계산 → 반경 갱신 → 수락/거부. 두 전략(`cauchy`, `dogleg`)을 같은 출발점에서 실행.
4. **반복 기록**: $(k, f_k, \lVert g_k\rVert, \Delta_k, \rho_k, \text{step type})$ 를 표로.
5. **시각화 1**: 등고선 위에 두 전략의 반복 경로를 겹쳐 그린다.
6. **시각화 2**: 반복수 대 $\lVert g_k\rVert$ (semilog) 로 수렴 속도 대비.
7. **해석**: dogleg(뉴턴 활용)의 빠른 수렴 vs Cauchy(최급강하급)의 느림, 반경 적응의 역할.


In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ---- Rosenbrock: f, gradient, Hessian ----
def f(p):
    x, y = p
    return (1.0 - x)**2 + 100.0*(y - x**2)**2

def grad(p):
    x, y = p
    return np.array([-2.0*(1.0 - x) - 400.0*x*(y - x**2),
                     200.0*(y - x**2)])

def hess(p):
    x, y = p
    return np.array([[2.0 - 400.0*(y - 3.0*x**2), -400.0*x],
                     [-400.0*x, 200.0]])

def is_pd(B):
    try:
        np.linalg.cholesky(B); return True
    except np.linalg.LinAlgError:
        return False

def model_pred_reduction(g, B, p):
    # predicted reduction = m(0) - m(p) = -(g.p + 0.5 p.B.p)
    return -(g @ p + 0.5 * p @ (B @ p))


In [2]:
# ---- Cauchy point ----
def cauchy_point(g, B, Delta):
    gnorm = np.linalg.norm(g)
    if gnorm == 0.0:
        return np.zeros_like(g), "cauchy"
    gBg = g @ (B @ g)
    if gBg <= 0.0:
        tau = 1.0
    else:
        tau = min(gnorm**3 / (Delta * gBg), 1.0)
    return -tau * (Delta / gnorm) * g, "cauchy"

# ---- boundary intersection: largest tau>=0 with ||z+tau d||=Delta ----
def _to_boundary(z, d, Delta):
    a = d @ d
    b = 2.0 * (z @ d)
    c = (z @ z) - Delta**2
    disc = max(b*b - 4.0*a*c, 0.0)
    return (-b + np.sqrt(disc)) / (2.0*a)

# ---- Dogleg step (falls back to Cauchy if B not PD) ----
def dogleg_step(g, B, Delta):
    if not is_pd(B):
        return cauchy_point(g, B, Delta)
    pB = -np.linalg.solve(B, g)          # full Newton step
    if np.linalg.norm(pB) <= Delta:
        return pB, "dogleg-newton"
    gBg = g @ (B @ g)
    pU = -(g @ g) / gBg * g              # steepest-descent minimizer
    if np.linalg.norm(pU) >= Delta:      # even pU outside -> scale to boundary
        return (Delta / np.linalg.norm(pU)) * pU, "dogleg-cauchy"
    tau = _to_boundary(pU, pB - pU, Delta)   # tau in [0,1] on second leg
    return pU + tau * (pB - pU), "dogleg-leg2"


In [3]:
# ---- Trust-region driver ----
def trust_region(x0, step_kind, Delta0=1.0, Delta_max=100.0,
                 eta=0.1, tol=1e-8, maxit=10000):
    x = np.array(x0, dtype=float)
    Delta = Delta0
    rows = []
    for k in range(maxit):
        g = grad(x); B = hess(x); gn = np.linalg.norm(g)
        rows.append((k, f(x), gn, Delta))
        if gn < tol:
            break
        if step_kind == "cauchy":
            p, _ = cauchy_point(g, B, Delta)
        else:
            p, _ = dogleg_step(g, B, Delta)
        pred = model_pred_reduction(g, B, p)
        ared = f(x) - f(x + p)
        rho = ared / pred if pred > 0 else -np.inf
        pnorm = np.linalg.norm(p)
        if rho < 0.25:
            Delta = 0.25 * Delta
        elif rho > 0.75 and pnorm >= 0.99*Delta:
            Delta = min(2.0*Delta, Delta_max)
        if rho > eta:
            x = x + p
    hist = pd.DataFrame(rows, columns=["k", "f", "||grad||", "Delta"])
    return x, hist

x0 = (-1.2, 1.0)
x_dl, hist_dl = trust_region(x0, "dogleg")
x_ca, hist_ca = trust_region(x0, "cauchy", maxit=20000)

print(f"dogleg : iters={len(hist_dl)-1:5d}  x*={x_dl}  f={f(x_dl):.3e}  ||g||={np.linalg.norm(grad(x_dl)):.3e}")
print(f"cauchy : iters={len(hist_ca)-1:5d}  x*={x_ca}  f={f(x_ca):.3e}  ||g||={np.linalg.norm(grad(x_ca)):.3e}")
print(f"true minimizer = (1, 1), f* = 0")


dogleg : iters=   24  x*=[1. 1.]  f=8.589e-22  ||g||=1.269e-09
cauchy : iters=19490  x*=[0.99999999 0.99999998]  f=7.894e-17  ||g||=9.998e-09
true minimizer = (1, 1), f* = 0


In [4]:
# ---- Iteration table: dogleg (전 구간), cauchy (처음/끝 일부) ----
pd.set_option("display.float_format", lambda v: f"{v:.6e}")
view = hist_dl.copy()
view.insert(0, "method", "dogleg")
print("[Dogleg] full iteration history")
view


[Dogleg] full iteration history


,method,k,f,||grad||,Delta
0,dogleg,0,2.420000e+01,2.328677e+02,1.000000e+00
1,dogleg,1,4.731884e+00,4.639426e+00,1.000000e+00
2,dogleg,2,4.731884e+00,4.639426e+00,2.500000e-01
3,dogleg,3,4.302044e+00,4.832101e+00,5.000000e-01
4,dogleg,4,3.601675e+00,1.842248e+01,1.000000e+00
5,dogleg,5,2.918501e+00,1.793994e+01,1.000000e+00
6,dogleg,6,2.266976e+00,9.333244e+00,1.000000e+00
7,dogleg,7,1.898640e+00,1.461423e+01,1.000000e+00
8,dogleg,8,1.316693e+00,3.331186e+00,1.000000e+00
9,dogleg,9,1.316693e+00,3.331186e+00,2.500000e-01


In [5]:
print("[Cauchy point] head & tail (느린 수렴 확인)")
ca = hist_ca.copy()
display_rows = pd.concat([ca.head(6), ca.tail(4)])
display_rows.insert(0, "method", "cauchy")
display_rows


[Cauchy point] head & tail (느린 수렴 확인)


,method,k,f,||grad||,Delta
0,cauchy,0,2.420000e+01,2.328677e+02,1.000000e+00
1,cauchy,1,4.567782e+00,3.094498e+01,1.000000e+00
2,cauchy,2,4.128383e+00,1.948900e+00,1.000000e+00
3,cauchy,3,4.117760e+00,4.218424e+00,1.000000e+00
4,cauchy,4,4.107323e+00,1.964111e+00,1.000000e+00
5,cauchy,5,4.097064e+00,4.105997e+00,1.000000e+00
19487,cauchy,19487,7.934395e-17,1.309275e-08,2.500000e-01
19488,cauchy,19488,7.920827e-17,1.001544e-08,2.500000e-01
19489,cauchy,19489,7.907282e-17,1.307043e-08,2.500000e-01
19490,cauchy,19490,7.893761e-17,9.998323e-09,2.500000e-01


In [6]:
# ---- Visualization 1: contour + iterate paths ----
def path_of(x0, kind, maxit=10000):
    x = np.array(x0, float); Delta = 1.0; pts = [x.copy()]
    for k in range(maxit):
        g = grad(x); B = hess(x)
        if np.linalg.norm(g) < 1e-8: break
        if kind == "cauchy":
            p, _ = cauchy_point(g, B, Delta)
        else:
            p, _ = dogleg_step(g, B, Delta)
        pred = model_pred_reduction(g, B, p); ared = f(x) - f(x+p)
        rho = ared/pred if pred > 0 else -np.inf
        pn = np.linalg.norm(p)
        if rho < 0.25: Delta *= 0.25
        elif rho > 0.75 and pn >= 0.99*Delta: Delta = min(2*Delta, 100.0)
        if rho > 0.1:
            x = x + p; pts.append(x.copy())
    return np.array(pts)

P_dl = path_of(x0, "dogleg")
P_ca = path_of(x0, "cauchy", maxit=4000)

xx = np.linspace(-1.6, 1.4, 400); yy = np.linspace(-0.6, 1.6, 400)
XX, YY = np.meshgrid(xx, yy)
ZZ = (1 - XX)**2 + 100*(YY - XX**2)**2

fig, ax = plt.subplots(figsize=(7.2, 5.4))
cs = ax.contour(XX, YY, ZZ, levels=np.logspace(-0.5, 3.5, 18), cmap="Greys", linewidths=0.6)
ax.plot(P_ca[:,0], P_ca[:,1], ".-", ms=3, lw=0.7, color="tab:orange",
        label=f"Cauchy point path ({len(P_ca)-1} steps shown)")
ax.plot(P_dl[:,0], P_dl[:,1], "o-", ms=4, lw=1.2, color="tab:blue",
        label=f"Dogleg path ({len(P_dl)-1} steps)")
ax.plot(1, 1, "r*", ms=15, label="minimizer (1,1)")
ax.plot(*x0, "ks", ms=7, label="start (-1.2, 1)")
ax.set_xlabel("x"); ax.set_ylabel("y")
ax.set_title("Trust-region iterate paths on Rosenbrock")
ax.legend(loc="upper left", fontsize=8)
plt.tight_layout(); plt.savefig("tr_paths.png", dpi=110); plt.show()


In [7]:
# ---- Visualization 2: convergence of ||grad|| ----
fig, ax = plt.subplots(figsize=(7.2, 5.0))
ax.semilogy(hist_dl["k"], hist_dl["||grad||"], "o-", ms=4, lw=1.2,
            color="tab:blue", label="dogleg")
ax.semilogy(hist_ca["k"], hist_ca["||grad||"], "-", lw=1.0,
            color="tab:orange", label="Cauchy point")
ax.set_xlabel("iteration k"); ax.set_ylabel(r"$\||\nabla f(x_k)\||$")
ax.set_title("Gradient-norm convergence: dogleg vs Cauchy point")
ax.set_xlim(0, min(len(hist_ca), 200))
ax.grid(True, which="both", ls=":", alpha=0.5)
ax.legend()
plt.tight_layout(); plt.savefig("tr_convergence.png", dpi=110); plt.show()


## 4. 결과 해석

1. **Dogleg**: 뉴턴 스텝 정보를 신뢰영역 안에서 활용하므로, 골짜기에 진입한 뒤에는 거의 완전 뉴턴 스텝
   ($\rho\approx1$, 반경이 충분히 커진 상태)을 밟아 **수십 회 이내**에 $(1,1)$ 로 2차 수렴한다.
2. **Cauchy 점**: 매 반복 최급강하 방향으로만 움직여 Rosenbrock의 좁고 굽은 골짜기를 **지그재그**로 기어간다.
   같은 허용오차에 도달하는 데 dogleg보다 수십~수백 배 많은 반복이 필요하다(표/그래프에서 확인).
3. **반경 적응의 역할**: 출발점처럼 모델이 부정확한 곳에서는 $\rho$ 가 작아 반경이 줄고(보수적), 골짜기에서
   모델이 정확해지면 $\rho\to1$ 로 반경이 커져 큰 스텝을 허용한다 — 직선탐색의 스텝길이 역할을 반경이 대신한다.
4. **경로 그림**: dogleg 경로는 골짜기를 가로질러 곧장 최소점으로, Cauchy 경로는 골짜기 바닥을 따라
   촘촘히 누적된다.

### 결론
> **신뢰영역의 위력은 "스텝 방향과 길이를 모델 신뢰도($\rho$)로 동시에 조절"하는 데 있다** —
> 같은 골격 안에서 dogleg(뉴턴 활용)은 2차 수렴, Cauchy 점(최급강하)은 1차 수렴으로 갈린다.

### 다음 문제 연결
- **CE 13.7.2**: dogleg은 $B\succ0$ 를 요구한다. 다음 문제에서는 **부분문제를 정확히** 풀어
  (Levenberg–Marquardt 모수 $\lambda$ 의 secular 방정식) **음의 곡률**과 **hard case** 까지 처리한다.
